# Data Description
- Author:
- Date first created: YYYYMMDD
- Background: 簡述資料內容
- Unit of observation:
- Sample period:
- Known issues:
- Definition for each variable:
    -
    -
    -
- Data path: `data/processed/<OUTPUT_FILE_NAME.csv>` (the exact file this notebook reads)
- Source data: `<list every exact data/raw/... or upstream data/processed/... input>`
- Processing code: `data/code/<PROCESSING_NAME.ipynb-or-Rmd>` (record the actual producer, or explain if supplied preprocessed)
- Role: `<analysis | train | test; use the actual role, not a filename guess>`
- Verification status: `<pending; only update after reopening this exact file and checking it>`
- SHA-256: `<pending; fingerprint of this processed file after successful checks>`
- Downstream analysis input: `data/processed/<OUTPUT_FILE_NAME.csv>` (must equal Data path)

Keep these paths and the `PROJECT.md` Data Artifacts row consistent. The source
and processing links describe lineage; later finding notebooks read this exact
processed file. For train/test data, document each file and its role separately.
Replace placeholders and adapt reader options to preserve the intended types.


In [ ]:
from pathlib import Path
import hashlib
import pandas as pd


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "AGENTS.md").is_file() and (candidate / "templates").is_dir():
            return candidate
    raise FileNotFoundError("Open this notebook within the student project containing AGENTS.md and templates/.")


PROJECT_ROOT = find_project_root()


def project_file(relative_path, area, must_exist=True):
    # Explicit project-relative paths only; never select the first matching file.
    rel = Path(relative_path)
    if rel.is_absolute() or ".." in rel.parts or "<" in str(rel):
        raise ValueError(f"Replace the placeholder with an exact project-relative path: {relative_path}")
    path = (PROJECT_ROOT / rel).resolve()
    allowed = (PROJECT_ROOT / "data" / area).resolve()
    if allowed == PROJECT_ROOT or not allowed.is_relative_to(PROJECT_ROOT):
        raise ValueError(f"data/{area}/ resolves outside the project root.")
    if not path.is_relative_to(allowed):
        raise ValueError(f"Expected a file inside data/{area}/: {relative_path}")
    if must_exist and not path.is_file():
        raise FileNotFoundError(path)
    return path


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def verify_input_sha256(path, expected):
    # Bind this run to the exact file inspected when the notebook was prepared.
    if (not isinstance(expected, str) or len(expected) != 64
            or any(char not in "0123456789abcdefABCDEF" for char in expected)):
        raise ValueError("Fill the expected SHA-256 from the inspected input before loading data.")
    actual = sha256_file(path)
    if actual != expected.lower():
        raise ValueError(f"Input changed: identify the new version before loading {path.name}.")
    return actual


## Change for Your Own Data

Use the exact project-relative Data path above. A missing path stops execution;
do not substitute another CSV or use the template/example data. The CSV reader
below is a scaffold: configure dtypes, date parsing, missing codes, and the reader
for the actual stored format. Fill the expected SHA-256 from the exact file
inspected while preparing this notebook; placeholders or a changed file stop
data loading. Documentation stays pending until executed.


In [ ]:
INPUT_RELATIVE_PATH = "data/processed/<OUTPUT_FILE_NAME.csv>"
input_data_file = project_file(INPUT_RELATIVE_PATH, "processed")
EXPECTED_COLUMNS = ["<COLUMN_NAME>"]  # Replace with the ordered schema from the actual producer.
EXPECTED_ROWS = None  # Set when the processing record supplies an expected row count.
EXPECTED_SHA256 = "<INSPECTED_INPUT_SHA256>"  # Bind to the actual file inspected for this notebook.


## Summary


In [ ]:
input_fingerprint = verify_input_sha256(input_data_file, EXPECTED_SHA256)
df = pd.read_csv(input_data_file)  # Configure dtype/parse_dates/na_values for the actual file.
print({"data_path": INPUT_RELATIVE_PATH, "rows": len(df),
       "schema": {column: str(dtype) for column, dtype in df.dtypes.items()},
       "sha256": sha256_file(input_data_file)})


### Sample


In [ ]:
df.head(10)


### Summary Stats


In [ ]:
df.describe(include="all")


### Assertions


In [ ]:
# Replace the schema placeholder; add checks for the actual keys, ranges,
# missingness, units, merge coverage, and train/test separation as applicable.
if not EXPECTED_COLUMNS or any("<" in name for name in EXPECTED_COLUMNS):
    raise ValueError("Fill the expected schema and dataset-specific assertions before verification.")
assert list(df.columns) == EXPECTED_COLUMNS, "Processed schema differs from the documented schema."
if EXPECTED_ROWS is not None:
    assert len(df) == EXPECTED_ROWS, "Processed row count differs from its processing record."
actual_sha256 = sha256_file(input_data_file)
assert actual_sha256 == input_fingerprint, "File changed during documentation: inspect the new version."
print({"data_path": INPUT_RELATIVE_PATH, "rows": len(df), "columns": list(df.columns),
       "sha256": actual_sha256, "status": "configured checks passed"})
# Update Data Description and the PROJECT.md Data Artifacts row only from the
# real results. Store the same exact path and verified SHA-256 in downstream
# finding code; do not rename/copy this file merely to match another example.
